In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())
exec(open(os.path.join(path_git, 'Data', 'BLS', 'config', 'BLS Functions.py')).read())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

***

BLS

***

In [ ]:
## Importing ---
indicator_name = 'Jobs_1'
plot_name = 'monthly_line_test'
export = False

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)



## Organizing ---

df_plot = df_jobs.copy()

# Growth rate
df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])

# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])
df_plot['Jobs_GR'] = round(df_plot['Jobs_GR'], 2)
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})


display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

fig = px.line(df_plot, x='date_', y='Growth Rate', color='Groups', color_discrete_map=color_map)

title = 'Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas'
fig.update_layout(xaxis_title = 'Date')
fig.update_yaxes(tick0=0, dtick=2, ticksuffix='%')
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")


plot_agol(export=export)



In [ ]:
## Importing ---
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Oragnizing ---

df_plot = df_jobs.copy()
month = '09'

df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
df_plot = df_plot.reset_index(drop = True)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])

conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2000, 2007, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2008, 2011, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2012, 2019, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2020, 2020, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2021, 2024, 1)))
             ]
choices = ["Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2024)"]
df_plot["Period"] = np.select(conditions, choices)

df_plot = df_plot.dropna()
df_plot1 = df_plot.groupby(['Groups', 'Period'], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2 = df_plot.groupby(['Groups'          ], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2['Period'] = 'Total<br>(2000-2024)'
df_plot = pd.concat([df_plot1, df_plot2])
categories = ["Total<br>(2000-2024)", "Pre Recession<br>(2000-2008)", "Recession<br>(2008-2011)", "Post Recession<br>(2011-2020)", "Covid<br>(2020)", "Post Covid<br>(2020-2024)"]
df_plot['Period_Sort'] = pd.Categorical(df_plot['Period'], categories)
df_plot = df_plot.sort_values(by = ['Groups', 'Period_Sort'], ascending = [True, True])
df_plot = df_plot.drop(['Period_Sort'], axis = 1)
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate', 'Period':'Time Period'})

display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
# df_plot = df_plot.rename(columns = {'Group':'Geography'})
fig1 = px.bar(df_plot, x='Time Period', y='Growth Rate'
             , color='Groups'
             , color_discrete_map=color_map
             , barmode='group'
             , hover_name = 'Time Period')
fig1.update_yaxes(tick0=0, dtick=2, ticksuffix='%')
fig1.update_xaxes(tickangle=0)
fig1.update_layout(xaxis_title=None, xaxis=dict(tickfont = dict(size=11)))

title = 'Annual Job Growth Comparison: Sacramento, National, and other Mid-Sized Metro Areas (September)'
fig1.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig1.update_traces(hovertemplate="Growth Rate: %{y}")
fig1.show()


## Importing ---
indicator_name = 'Jobs_1'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Organizing ---

df_plot = df_jobs.copy()
month = '09'

df_plot = df_plot[df_plot['Sector'] == 'All']
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
df_plot = df_plot.reset_index(drop = True)

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])


df_plot = df_plot.dropna()
df_plot = df_plot.sort_values(by = ['Groups', 'date_'], ascending = [True, True])
df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})

df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
df_plot = df_plot.rename(columns = {'Groups':'Geography'})

display(df_plot.head())


## Plotting ---

color_map = {
     "SACOG":"#9DC209",
     "National": "#1F45FC",
     "Peer MSA": "#1E90FF"
}

fig2 = px.line(df_plot, x = 'date_', y = 'Growth Rate', color = 'Geography', color_discrete_map=color_map)
fig2.update_yaxes(tick0=0, dtick=1)
fig2.add_vline(x = '2009-01-01', line_dash = 'dash')
fig2.add_vline(x = '2012-01-01', line_dash = 'dash')
fig2.add_vline(x = '2020-01-01', line_dash = 'dash')
fig2.add_vline(x = '2021-03-01', line_dash = 'dash')
fig2.add_annotation(x='2005-01-01', y = -6.5, text="Pre-Recession" , showarrow= False)
fig2.add_annotation(x='2010-07-01', y = -6.5, text="Recession"     , showarrow= False)
fig2.add_annotation(x='2016-01-01', y = -6.5, text="Post-Recession", showarrow= False)
fig2.add_annotation(x='2020-07-20', y = -6.5, text="Covid"         , showarrow= False)
fig2.add_annotation(x='2022-09-01', y = -6.5, text="Post-Covid"    , showarrow= False)
fig2.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')
fig2.update_xaxes(dtick="M24", tickformat="%Y", ticklabelmode="period")
fig2.update_layout(xaxis_title  = 'Year')


title = 'Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas (September)'
fig2.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
fig2.update_traces(hovertemplate="Year: %{x}<br>Growth Rate: %{y}")


fig2.show()

In [ ]:
# https://stackoverflow.com/questions/56727843/how-can-i-create-subplots-with-plotly-express


## Plotting ---

indicator_name = 'Jobs_1'
plot_name = 'side_by_side_bar_line_test'
export = False


fig1_traces = []
fig2_traces = []


for trace in range(len(fig1["data"])):
    fig1_traces.append(fig1["data"][trace])

for trace in range(len(fig2["data"])):
    fig2["data"][trace]['showlegend'] = False
    fig2_traces.append(fig2["data"][trace])

# Create a 1x2 subplot
fig = sp.make_subplots(rows = 1, cols = 2
                       , subplot_titles=('<span style="font-size: 13px;">By Time Period (September to September)</span>', 
                                         '<span style="font-size: 13px;">By Year (September to September)</span>')
                      )

# Get the Express fig broken down as traces and add the traces to the proper plot within the subplot
for traces in fig1_traces:
    fig.append_trace(traces, row = 1, col = 1)
for traces in fig2_traces:
    fig.append_trace(traces, row = 1, col = 2)

# fig.update_layout(legend_title=None, title='Monthly Job Growth: Sacramento and other Mid-Sized Metro Areas (by Presidential Administration) (November)')
fig.add_vline(x = '2009-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2012-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2020-01-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_vline(x = '2021-03-01', line_dash = 'dash', row=1, col=2, line_color = 'gray')
fig.add_annotation(x='2005-01-01', y = -7, text='<span style="font-size: 7px;">Pre-Recession</span>' , showarrow= False, row=1, col=2)
fig.add_annotation(x='2010-07-01', y = -7, text='<span style="font-size: 7px;">Recession</span>'     , showarrow= False, row=1, col=2)
fig.add_annotation(x='2016-01-01', y = -7, text='<span style="font-size: 7px;">Post-Recession</span>', showarrow= False, row=1, col=2)
fig.add_annotation(x='2020-07-20', y = -7, text='<span style="font-size: 7px;">Covid</span>'         , showarrow= False, row=1, col=2)
fig.add_annotation(x='2022-07-01', y = -7, text='<span style="font-size: 7px;">Post-Covid</span>'    , showarrow= False, row=1, col=2)
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray', row=1, col=2)
            

title='Job Growth Comparison: Sacramento, National, and other Mid-Sized Metro Areas'
fig.update_yaxes(tick0=0, dtick=2, ticksuffix='%', range = [-7,7])
fig.update_xaxes(tickangle=0)
fig.update_xaxes(showticklabels=False, showgrid=False, row=1, col=2)

# fig.show()
plot_agol(export=export)


Seasonal Decomposition

In [ ]:
## Importing ---
indicator_name = 'Jobs_1'
plot_name = 'monthly_line_test'
export = False

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)



## Organizing ---

df_plot = df_jobs.copy()

# Growth rate
df_plot = df_plot[df_plot['Sector'] != 'All']
df_plot = df_plot.sort_values(['Geography', 'Sector', 'date_'], ascending = [True, True, True])

df_plot = df_plot.groupby(['date_', 'Geography', 'Sector'], as_index = False).agg(Value = ('Value', 'sum'))
df_plot = df_plot.sort_values(['Geography', 'date_', 'Sector'], ascending = [True, True, True])
df_plot = df_plot[df_plot['Geography'] == 'Sacramento--Roseville--Arden-Arcade, CA']
df_plot = df_plot[df_plot['Sector'] == 'Government']
df_plot['moving_avg'] = df_plot['Value'].rolling(window=12).mean()


display(df_plot.head())


# ## Plotting ---


fig = px.line(df_plot, x='date_', y='Value', markers = False, color = 'Sector')
fig['data'][0]['line']['color']='#1E90FF'

fig.add_trace(go.Scatter(x=df_plot["date_"], y=df_plot['moving_avg']
                         , name = 'Seasonal Decomposition'
                         , line=go.scatter.Line(color="gray", dash="dot")
                        ))

title = 'Monthly Job Growth: Sacramento Goverment'
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")


plot_agol(export=export)



In [ ]:
df_jobs['Sector'].unique()

In [ ]:

## Importing ---
indicator_name = 'Jobs_2'

file_name1 = f"{indicator_name} MSA BLS SM.xlsx"
file_name2 = f"{indicator_name} National BLS CE.xlsx"

df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_name1), sheet_name = 'MSA')
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_name2), sheet_name = 'National')

df_msa = df_msa.rename(columns = {'MSA':'Geography'})
df_nat = df_nat.rename(columns = {'MSA':'Geography'})

df_jobs = pd.concat([df_msa, df_nat])
df_jobs = df_jobs.rename(columns = {'Total Jobs':'Value'})
df_jobs = df_jobs.reset_index(drop = True)


## Organizing ---

for sector in df_jobs['Sector'].unique():
    
    df_plot = df_jobs.copy()
    month = '09'
    
    df_plot = df_plot[df_plot['Sector'] == sector]
    df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
    df_plot = df_plot[df_plot['date_'].str.contains(f'{month}-01')]
    df_plot = df_plot[~df_plot['date_'].str.contains('2000-01-01')]
    df_plot = df_plot[~df_plot['date_'].str.contains(f'20{month}-01-01')]
    df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
    df_plot.loc[df_plot['date_'] == f'2000-{month}-01', 'Jobs_GR'] = np.nan
    df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
    df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])
    df_plot = df_plot[~df_plot['Jobs_GR'].isna()]
    df_plot = df_plot.reset_index(drop = True)
    
    wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average
    
    conditions = [   
           df_plot['Geography'].str.contains('Sac|Yuba')
        , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
        ,  df_plot['Geography'].str.contains('National')
                 ]
    choices = ['SACOG', 'Peer MSA', 'National']
    df_plot['Groups'] = np.select(conditions, choices)
    df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
    df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])
    
    
    df_plot = df_plot.dropna()
    df_plot = df_plot.sort_values(by = ['Groups', 'date_'], ascending = [True, True])
    df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})
    
    df_plot['Growth Rate'] = round(df_plot["Growth Rate"], 2)
    df_plot = df_plot.rename(columns = {'Groups':'Geography'})
    
    display(df_plot.head())
    
    
    ## Plotting ---
    
    color_map = {
         "SACOG":"#9DC209",
         "National": "#1F45FC",
         "Peer MSA": "#1E90FF"
    }
    
    fig = px.line(df_plot, x = 'date_', y = 'Growth Rate', color = 'Geography', color_discrete_map=color_map)
    fig.update_yaxes(tick0=0, dtick=1)
    fig.add_vline(x = '2009-01-01', line_dash = 'dash')
    fig.add_vline(x = '2012-01-01', line_dash = 'dash')
    fig.add_vline(x = '2020-01-01', line_dash = 'dash')
    fig.add_vline(x = '2021-03-01', line_dash = 'dash')
    fig.add_annotation(x='2005-01-01', y = -6.5, text="Pre-Recession" , showarrow= False)
    fig.add_annotation(x='2010-07-01', y = -6.5, text="Recession"     , showarrow= False)
    fig.add_annotation(x='2016-01-01', y = -6.5, text="Post-Recession", showarrow= False)
    fig.add_annotation(x='2020-07-20', y = -6.5, text="Covid"         , showarrow= False)
    fig.add_annotation(x='2022-09-01', y = -6.5, text="Post-Covid"    , showarrow= False)
    fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')
    fig.update_xaxes(dtick="M24", tickformat="%Y", ticklabelmode="period")
    fig.update_layout(xaxis_title  = 'Year')
    
    
    title = f'Monthly {sector} Job Growth: Sacramento and other Mid-Sized Metro Areas (September)'
    fig.update_layout(legend_title=None, title=title, template=template, font_family=font_family)
    fig.update_traces(hovertemplate="Year: %{x}<br>Growth Rate: %{y}")

    fig.show()


***

Census

***

In [ ]:
# Set Indicator
indicator_name = 'Pop_3'
plot_name = 'race_ethnicity_test'
export = False


## Importing ---

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')



## Organizing ---

df_plot = df_mpo.copy()


df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot[df_plot['Race_Ethnicity'] != 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Race_Ethnicity'], [
    'American Indian or Alaska Native (NH)'
    , 'Native Hawaiian or other Pacific Islander (NH)'
    , 'Some other race (NH)'
    , 'Two or more races (NH)'
    , 'Black or African American (NH)'
    , 'Asian (NH)'
    , 'Hispanic or Latino'
    , 'White (NH)'
])
    
df_plot = df_plot.sort_values('Sort', ascending=False)
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---


color_map  = {
    'American Indian or Alaska Native (NH)': '#A97142'
    , 'Native Hawaiian or other Pacific Islander (NH)': '#006A4E'
    , 'Some other race (NH)': '#7E587E'
    , 'Two or more races (NH)': '#1F45FC'
    , 'Asian (NH)': '#9DC209'
    , 'Black or African American (NH)': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}




fig = px.area(df_plot, x='Year', y='Percentage', color='Race_Ethnicity', color_discrete_map=color_map)


title = '<b>Race and Ethnicity</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=10, range = [0, 105], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1, range=[2009.1, 2022])
fig.update_traces(hovertemplate='%{y}', line_width=0)
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)

In [ ]:
# Set Indicator
indicator_name = 'Pop_4'
plot_name = 'age_groups_test'
export = False



## Importing ---

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')



## Organizing ---

df_plot = df_mpo.copy()


df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
    'Under 18'
    , '18 to 64'
    , '65+'
])
    
df_plot = df_plot.sort_values('Sort', ascending=True)
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---
color_map = {
    'Under 18': '#1F45FC'
    , '18 to 64': '#1E90FF'
    , '65+': '#9DC209'
}


fig = px.area(df_plot, x='Year', y='Percentage', color='Variable', color_discrete_map=color_map)


title = '<b>Population Age Distribution</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=10, range = [0, 105], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1, range=[2009.1, 2022])
fig.update_traces(hovertemplate='%{y}', line_width=0)
fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator_name = 'Labor_1'
plot_name = 'participation'
export = False


## Importing ---

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')


## Organizing ---

df_plot = df_mpo.copy()

race_ethnicity = ['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)']
df_plot = df_plot[df_plot['Race_Ethnicity'].isin(race_ethnicity)]
df_plot = df_plot[df_plot['Year'] == 2022]
df_plot['Percentage'] = round(df_plot['Percentage'], 1)


df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
    'Employed'
    , 'Not in Labor Force'
    , 'Unemployed'
])

df_plot = df_plot.sort_values('Sort', ascending = True)
df_plot = df_plot.drop(['Sort'], axis = 1)

display(df_plot.head())


df_plot = df_plot.reset_index(drop = True)


display(df_plot.head())


## Plotting ---

color_map = {
         "Employed":"#1E90FF",
         "Unemployed": "#9DC209",
         "Not in Labor Force": "#1F45FC"
}

fig = px.sunburst(df_plot, path=['Race_Ethnicity', 'Variable'], values='Population'
             , color = 'Percentage'
            , color_continuous_scale='RdBu'
                  , color_continuous_midpoint=np.average(df_plot['Percentage'], weights=df_plot['Population'])
                 )



title = '<b>Working Age (16-64) Labor Force Participation by Race/Ethnicity, 2022</b>  <br><sup>6-County Sacramento Region</sup> '
# fig.update_yaxes(dtick=20, ticksuffix='%', range = [0,102])
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_traces(hovertemplate="%{y}")


plot_agol(export=export)


In [ ]:
# Set Indicator
indicator_name = 'Edu_1'
plot_name = 'education_rates_sunburst'
export = False


## Importing ---
indicator_name = 'Edu_1'

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')


## Organizing ---
df_plot = df_mpo.copy()

df_plot = df_plot[df_plot['Race_Ethnicity'].isin(['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)'])]
df_plot = df_plot[df_plot['Year'] == 2022]


df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
    'Total Less than high school diploma'
    , 'Total High school graduate or GED'
    , "Total Some college or associate's degree"
    , "Total Bachelor's degree or higher"
])
    
df_plot = df_plot.sort_values(['Sort'], ascending = [False])
df_plot = df_plot.drop(['Sort'], axis = 1)

df_plot['Percentage'] = round(df_plot['Percentage'], 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---
color_map  = {
    'Total Less than high school diploma':'#FBB117'
    , 'Total High school graduate or GED':'#9DC209'
    , "Total Some college or associate's degree":'#1E90FF'
    , "Total Bachelor's degree or higher":'#1F45FC'
    , '(?)':'lightgray'
}


fig = px.sunburst(df_plot, path=['Race_Ethnicity', 'Variable'], values='Population'
             , color = 'Variable'
            , color_discrete_map=color_map
                 )



title = '<b>Educational Attainment by Race/Ethnicity, 2022</b>  <br><sup>6-County Sacramento Region</sup> '
# fig.update_yaxes(tick0=0, dtick=10, ticksuffix='%')
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})


plot_agol(export=export)


In [ ]:
## Importing ---

estimate = 'ACS5'
indicator_name = 'Accessibility_4'

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')

df_acs_p = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_P.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})


df_acs_h = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_H.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})


df_cpi = pd.read_excel(os.path.join(path_config0, 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')

display(df_acs_p.head(), df_acs_h.head(), df_cpi.head())


## Organizing ---

df_acs_p = df_acs_p[df_acs_p['SPORDER'] == 1]
df_acs_p = df_acs_p[df_acs_p['JWTRNS'] != 'N/A, not a civillian in the labor force']

df_acs_h = df_acs_h.merge(df_acs_p.drop(['SPORDER', 'PWGTP'], axis = 1), on = ['State FIPS', 'PUMA', 'PUMA NAME', 'SERIALNO', 'year'], how = 'left')
df_acs_h = df_acs_h.dropna(subset = ['JWTRNS'])

df_cpi = df_cpi[['Year', 'IAF_2023']].rename(columns = {'Year':'year'})


df_acs_h = df_acs_h.merge(df_cpi, on = 'year', how = 'left')
df_acs_h['HINCP'] = df_acs_h['HINCP']*df_acs_h['ADJINC']*df_acs_h['IAF_2023']
df_acs_h = df_acs_h.drop(['IAF_2023', 'ADJINC'], axis = 1)

df_acs = df_acs_h.copy()

df_fips1 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS' 
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips2 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'PUMAcodes' 
                                , dtype = {'STATEFP': object, 'COUNTYFP': object})

df_fips1 = df_fips1[df_fips1['State FIPS'].isin(['06'])]
df_fips2 = df_fips2[df_fips2['STATEFP'   ].isin(['06'])]

df_fips2['PUMA5CE'] = df_fips2['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_fips2 = df_fips2[['STATEFP', 'COUNTYFP', 'PUMA5CE', 'Years']].drop_duplicates()
df_fips2 = df_fips2.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS', 'PUMA5CE':'PUMA'})

df_acs['PUMA'] = df_acs['PUMA'].astype(str).apply('{:0>5}'.format)

df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MPO']]
# df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MSA']]

df_acs2 = df_acs[df_acs['year'].isin(sequence(2022, 2031, 1))].merge(df_fips2[df_fips2['Years'] == '2022-2031'], on = ['State FIPS', 'PUMA'])
df_acs1 = df_acs[df_acs['year'].isin(sequence(2012, 2021, 1))].merge(df_fips2[df_fips2['Years'] == '2012-2021'], on = ['State FIPS', 'PUMA'])
df_acs = pd.concat([df_acs1, df_acs2])

df_acs = df_acs.merge(df_fips1, on = ['State FIPS', 'County FIPS'])
df_acs = df_acs.drop(['SERIALNO', 'Years'], axis = 1)


df_income_brackets = pd.read_excel(os.path.join(path_config0, 'CA State Income Brackets by Household Size.xlsx'), sheet_name = 'Table')
df_income_brackets['County'].fillna(method='ffill', inplace = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' County.*'         , '' , regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace('\n'                , ' ', regex = True)
df_income_brackets['AMI'   ] = df_income_brackets['County'].str.extract('\$?([0-9,]+)[.%]?')
df_income_brackets['AMI'   ] = df_income_brackets['AMI'   ].str.replace(','                 , '' , regex = True)
df_income_brackets['County'] = df_income_brackets['County'].str.replace(' \$?([0-9,]+)[.%]?', '' , regex = True)
df_income_brackets = pd.melt(df_income_brackets
                              , id_vars = ['County', 'Income Bracket', 'AMI']
                              , var_name = 'NP'
                              , value_name = 'Income Threshold'
                            )
df_income_brackets = df_income_brackets[df_income_brackets['Income Bracket'].isin(['Low Income', 'Moderate Income'])]
df_income_brackets = df_income_brackets.pivot_table(index = ['County', 'NP']
                                       , columns = 'Income Bracket'
                                       , values = 'Income Threshold').reset_index().rename(columns = {'County':'County Name'})

df_acs_jwmnp = df_acs.merge(df_income_brackets, on = ['County Name', 'NP'], how = 'left')


df_acs_jwmnp.loc[ df_acs_jwmnp['HINCP'] <= df_acs_jwmnp['Low Income']                                                              , 'Income Bracket'] = 'Low Income'
df_acs_jwmnp.loc[(df_acs_jwmnp['HINCP']  > df_acs_jwmnp['Low Income']) & (df_acs_jwmnp['HINCP'] <= df_acs_jwmnp['Moderate Income']), 'Income Bracket'] = 'Moderate Income'
df_acs_jwmnp.loc[ df_acs_jwmnp['HINCP']  > df_acs_jwmnp['Moderate Income']                                                         , 'Income Bracket'] = 'High Income'
df_acs_jwmnp.loc[ df_acs_jwmnp['NP'] == 0                                                                                          , 'Income Bracket'] = 'No data available'


df_rep = df_acs_jwmnp[['State FIPS', 'MPO', 'County Name', 'year', 'RAC1P', 'Income Bracket', 'WGTP', 'JWMNP']]
df_rep = df_rep.groupby(['State FIPS', 'MPO', 'year','Income Bracket', 'JWMNP'], as_index = False)['WGTP'].agg(sum)
display(df_rep.head())


list_keys = []
for list_ in df_rep.values:
    list_keys.append(tuple(list_[:-1]))

list_values = []
for list_ in df_rep.values:
    list_values.append(list_[-1])

dict_replicates = dict(zip(list_keys, list_values))

list_df = []
for tuple_ in tqdm(list(dict_replicates.keys())):
    
    row  = list(tuple_)
    wgtp = dict_replicates[tuple_]

    list_df.append(pd.concat([pd.DataFrame(row).T] * wgtp))

df_rep = pd.concat(list_df)
df_rep.columns = ['State FIPS', 'MPO',  'year', 'Income Bracket', 'JWMNP']
display(df_rep.head())


In [ ]:

## Plotting ---

indicator_name = 'Accessibility_4'
plot_name = 'histogram_test'
export = True


df_plot = df_rep[df_rep['year'].isin([2019, 2020, 2021])]
df_plot = df_plot[df_plot['Income Bracket'] != 'Moderate Income']

df_plot.loc[df_plot['Income Bracket'] == 'Low Income' , 'Income Bracket'] = 'Low<br>Income'
df_plot.loc[df_plot['Income Bracket'] == 'High Income', 'Income Bracket'] = 'High<br>Income'

fig = px.histogram(df_plot, x="JWMNP", facet_col='year', facet_row='Income Bracket', nbins=40, color_discrete_sequence=['#1F45FC'])#, opacity=0.8

fig['layout']['xaxis' ]['title']['text']=''
fig['layout']['xaxis2']['title']['text']='Commute Times (Minutes)'
fig['layout']['xaxis3']['title']['text']=''

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(tick0=0, dtick=15)
fig.update_yaxes(tick0=0, dtick=20000, tickformat = ',.0f', range = [0, 61000])
fig.update_traces(hovertemplate='Travel Time: %{x} minutes<br>Count: %{y:,.0f} households')
fig.update_traces(marker_line_width=0.1,marker_line_color="white")
title = 'Head of Household by Commute Time (Minutes)'
fig.for_each_yaxis(lambda y: y.update(title_text=''))

for annotation in fig['layout']['annotations']: 
    annotation['textangle']= 0


plot_agol(export=export)


In [ ]:
# scatter plot with year on x-axis, commute time on y-axis, colored by income (SACOG)
### THIS ONE IS NOT GOOD

# ## Plotting ---

# indicator_name = 'Accessibility_4'
# plot_name = 'test'
# export = False

# df_plot = df_rep.copy()
# # scatter plot with year on x-axis, commute time on y-axis, colored by income (SACOG)


# ## Plotting ---

# indicator_name = 'Accessibility_4'
# plot_name = 'test'
# export = False

# df_plot = df_rep.copy()

# df_plot = df_plot[df_plot['Income Bracket'] != 'Moderate Income']
# fig = px.scatter(df_plot, x="year", y="JWMNP",color="Income Bracket")

# # fig.update_xaxes(tick0=0, dtick=25)
# # fig.update_yaxes(tick0=0, dtick=20000, tickformat = ',.0f')
# fig.update_traces(hovertemplate='Travel Time: %{y}')
# title = 'Commute Time (Minutes) by Income Bracket'

# plot_agol(export=export)
# df_plot = df_plot[df_plot['Income Bracket'] != 'Moderate Income']
# fig = px.scatter(df_plot, x="year", y="JWMNP",color="Income Bracket")

# # fig.update_xaxes(tick0=0, dtick=25)
# # fig.update_yaxes(tick0=0, dtick=20000, tickformat = ',.0f')
# fig.update_traces(hovertemplate='Travel Time: %{y}')
# title = 'Commute Time (Minutes) by Income Bracket'

# plot_agol(export=export)


In [ ]:

## Importing ---

estimate = 'ACS5'
indicator_name = 'Accessibility_4'

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')

df_acs_p = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_P.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})

df_acs_h = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_H.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})


df_cpi = pd.read_excel(os.path.join(path_config0, 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')

display(df_acs_p.head(), df_acs_h.head(), df_cpi.head())


## Organizing ---

df_acs_p = df_acs_p[df_acs_p['SPORDER'] == 1]
df_acs_p = df_acs_p[df_acs_p['JWTRNS'] != 'N/A, not a civillian in the labor force']

df_acs_h = df_acs_h.merge(df_acs_p.drop(['SPORDER', 'PWGTP'], axis = 1), on = ['State FIPS', 'PUMA', 'PUMA NAME', 'SERIALNO', 'year'], how = 'left')
df_acs_h = df_acs_h.dropna(subset = ['JWTRNS'])

df_cpi = df_cpi[['Year', 'IAF_2023']].rename(columns = {'Year':'year'})


df_acs_h = df_acs_h.merge(df_cpi, on = 'year', how = 'left')
df_acs_h['HINCP'] = df_acs_h['HINCP']*df_acs_h['ADJINC']*df_acs_h['IAF_2023']
df_acs_h = df_acs_h.drop(['IAF_2023', 'ADJINC'], axis = 1)

df_acs = df_acs_h.copy()

df_fips1 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS' 
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips2 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'PUMAcodes' 
                                , dtype = {'STATEFP': object, 'COUNTYFP': object})

df_fips1 = df_fips1[df_fips1['State FIPS'].isin(['06'])]
df_fips2 = df_fips2[df_fips2['STATEFP'   ].isin(['06'])]

df_fips2['PUMA5CE'] = df_fips2['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_fips2 = df_fips2[['STATEFP', 'COUNTYFP', 'PUMA5CE', 'Years']].drop_duplicates()
df_fips2 = df_fips2.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS', 'PUMA5CE':'PUMA'})

df_acs['PUMA'] = df_acs['PUMA'].astype(str).apply('{:0>5}'.format)

df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MPO']]
# df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MSA']]

df_acs2 = df_acs[df_acs['year'].isin(sequence(2022, 2031, 1))].merge(df_fips2[df_fips2['Years'] == '2022-2031'], on = ['State FIPS', 'PUMA'])
df_acs1 = df_acs[df_acs['year'].isin(sequence(2012, 2021, 1))].merge(df_fips2[df_fips2['Years'] == '2012-2021'], on = ['State FIPS', 'PUMA'])
df_acs = pd.concat([df_acs1, df_acs2])

df_acs = df_acs.merge(df_fips1, on = ['State FIPS', 'County FIPS'])
df_acs = df_acs.drop(['SERIALNO', 'Years'], axis = 1)

display(df_acs.head())

In [ ]:
# scatter plot with income on x axis, commute time on y-axis, colored by county (one year at a time)



## Plotting ---

indicator_name = 'Accessibility_4'
plot_name = 'test'
export = True

df_plot = df_acs.copy()
df_plot = df_plot[df_plot['year'].isin([2019, 2021])]
df_plot = df_plot[df_plot['HINCP'] < 250000]
df_plot['HINCP'] = round(df_plot['HINCP'], -4)
df_plot['JWMNP'] = round(df_plot['JWMNP'])

df_plot = df_plot.groupby(['year', 'HINCP', 'JWMNP'], as_index = False)['WGTP'].sum()
# df_plot = df_plot.groupby(['County Name', 'HINCP', 'JWMNP'], as_index = False)['WGTP'].sum()

fig = px.scatter(df_plot, x="HINCP", y="JWMNP", size = 'WGTP', facet_col = 'year')
# fig = px.scatter(df_plot, x="HINCP", y="JWMNP", color="County Name", size = 'WGTP')


# fig.update_xaxes(tick0=0, dtick=25)
fig.update_xaxes(tick0=0, dtick=50000, tickformat = ',.0f', range = [-26000, 260000])
fig.update_yaxes(tick0=0, dtick=15, range = [0,150])
# fig.update_traces(hovertemplate='Travel Time: %{y}<br>Household Income: %{x}')
title = 'Commute Time (Minutes) by Household Income'

plot_agol(export=export)

In [ ]:
# scatter plot with income on x axis, commute time on y-axis, colored by county (one year at a time)
# scatter plot with year on x-axis, commute time on y-axis, size by income, colored by county
# scatter plot with year on x-axis, commute time on y-axis, colored by income (SACOG)

In [ ]:
# Set Indicator
indicator_name = 'Cost_3'
plot_name = 'vacancy_rate'
export = False


## Importing ---

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')


## Organizing ---

df_plot = df_mpo.copy()


df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']
df_plot = df_plot[df_plot['Variable'] == 'Vacant']
df_plot = df_plot.reset_index(drop = True)
df_plot['Percentage_ME'] = df_plot['Margin of Error']/(df_plot['Total']/(df_plot['Percentage']/100))
df_plot['Percentage'   ] = round(df_plot['Percentage'   ], 1)
df_plot['Percentage_ME'] = round(df_plot['Percentage_ME']*100, 1)

display(df_plot.head())


## Plotting ---

fig = go.Figure([
    go.Scatter(
        name='Measurement',
        x=df_plot['Year'],
        y=df_plot['Percentage'],
        mode='lines',
        line=dict(color='#1E90FF'),
        showlegend=False
    ),
    go.Scatter(
        name='Upper Bound',
        x=df_plot['Year'],
        y=df_plot['Percentage']+df_plot['Percentage_ME'],
        mode='lines',
        marker=dict(color="#444"),
        line=dict(width=0),
        showlegend=False
    ),
    go.Scatter(
        name='Lower Bound',
        x=df_plot['Year'],
        y=df_plot['Percentage']-df_plot['Percentage_ME'],
        marker=dict(color="#444"),
        line=dict(width=0),
        mode='lines',
        fillcolor='rgba(68, 68, 68, 0.3)',
        fill='tonexty',
        showlegend=False
    )
])


title = '<b>Unoccupied Housing Rate</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=0.5, ticksuffix='%', range = [1.75,5.25])
fig.update_xaxes(dtick=1)
fig.update_layout(hovermode="x")

plot_agol(export=export)


In [ ]:
# Set Indicator
indicator_name = 'Cost_3'
plot_name = 'vacancy_rate'
export = False


## Importing ---

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')


## Organizing ---

df_plot = df_mpo.copy()


df_plot = df_plot[df_plot['Race_Ethnicity'] == 'All']
df_plot = df_plot[df_plot['Variable'] == 'Vacant']
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())


## Plotting ---

fig = go.Figure([
    go.Scatter(
        name='Measurement',
        x=df_plot['Year'],
        y=df_plot['Total'],
        mode='lines',
        line=dict(color='#1E90FF'),
        showlegend=False
    ),
    go.Scatter(
        name='Upper Bound',
        x=df_plot['Year'],
        y=df_plot['Total']+df_plot['Margin of Error'],
        mode='lines',
        marker=dict(color="#444"),
        line=dict(width=0),
        showlegend=False
    ),
    go.Scatter(
        name='Lower Bound',
        x=df_plot['Year'],
        y=df_plot['Total']-df_plot['Margin of Error'],
        marker=dict(color="#444"),
        line=dict(width=0),
        mode='lines',
        fillcolor='rgba(68, 68, 68, 0.3)',
        fill='tonexty',
        showlegend=False
    )
])


title = '<b>Unoccupied Housing Rate</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_yaxes(dtick=10000, range = [0,51000], tickformat = ',.0f')
fig.update_xaxes(dtick=1, range = [2008.5, 2022.5])
fig.update_layout(hovermode="x")

plot_agol(export=export)


***

Zillow

***

In [ ]:

## Importing ---

# Set Indicator
indicator_name = 'Income_1'

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_inc = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')



# Set Indicator
indicator_name = 'Cost_1'
plot_name = 'cost_to_income_ratio'
export = True


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Annual')


## Organizing ---

df_cost = df_cost[df_cost['Region'] == 'SACOG Median']
df_cost = df_cost[['Year', 'Price']]

df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
df_cpi = df_cpi[['Year', 'IAF_' + str(2022)]]

df_cost = df_cost.merge(df_cpi, on = 'Year', how = 'left')
df_cost['Price'] = round(df_cost['Price']*df_cost['IAF_' + str(2022)])
df_cost = df_cost.drop(['IAF_' + str(2022)], axis = 1)


df_inc = df_inc[['MPO', 'Year', 'Race_Ethnicity', 'Regional Median Income']]


df_inc = df_inc.merge(df_cost, on='Year', how='left')
df_inc['Cost to Median Income Ratio'] = round(df_inc['Price']/df_inc['Regional Median Income'], 2)


df_plot = df_inc[df_inc['Race_Ethnicity'].isin(['Black or African American', 'Asian', 'Hispanic or Latino', 'White (NH)'])]
df_inc = df_inc[df_inc['Race_Ethnicity'] == 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Race_Ethnicity'], [
    'Asian'
    , 'Black or African American'
    , 'Hispanic or Latino'
    , 'White (NH)'
])
    
df_plot = df_plot.sort_values(['Year', 'Sort'], ascending=[False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---


color_map  = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}


fig = px.line(df_plot, x='Year', y='Cost to Median Income Ratio', color='Race_Ethnicity', color_discrete_map=color_map, markers=True)

fig.add_trace(go.Scatter(x=df_inc["Year"], y=df_inc['Cost to Median Income Ratio']
                         , name = 'Average'
                         , line=go.scatter.Line(color="#2C3539", dash="dot")
                        ))

title = '<b>Housing Cost to Median Household Income Ratio</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=1, range = [0, 9.1])
fig.update_xaxes(tick0=0, dtick=1, range=[2008.5, 2022.5])
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)

In [ ]:

## Importing ---

# Set Indicator
indicator_name = 'Income_1'

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_inc = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')



# Set Indicator
indicator_name = 'Cost_2'
plot_name = 'rent_to_income_ratio'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Annual')


## Organizing ---

df_cost = df_cost[df_cost['Region'] == 'SACOG Median']
df_cost = df_cost[['Year', 'Price']]

df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
df_cpi = df_cpi[['Year', 'IAF_' + str(2022)]]

df_cost = df_cost.merge(df_cpi, on = 'Year', how = 'left')
df_cost['Price'] = round(df_cost['Price']*df_cost['IAF_' + str(2022)])
df_cost = df_cost.drop(['IAF_' + str(2022)], axis = 1)


df_inc = df_inc[['MPO', 'Year', 'Race_Ethnicity', 'Regional Median Income']]


df_inc = df_inc.merge(df_cost, on='Year', how='left')
df_inc['Cost to Median Income Ratio'] = round(df_inc['Price']/(df_inc['Regional Median Income']/12), 2)


df_plot = df_inc[df_inc['Race_Ethnicity'].isin(['Black or African American', 'Asian', 'Hispanic or Latino', 'White (NH)'])]
df_inc = df_inc[df_inc['Race_Ethnicity'] == 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Race_Ethnicity'], [
    'Asian'
    , 'Black or African American'
    , 'Hispanic or Latino'
    , 'White (NH)'
])
    
df_plot = df_plot.sort_values(['Year', 'Sort'], ascending=[False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---


color_map  = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}


fig = px.line(df_plot, x='Year', y='Cost to Median Income Ratio', color='Race_Ethnicity', color_discrete_map=color_map, markers=True)

fig.add_trace(go.Scatter(x=df_inc["Year"], y=df_inc['Cost to Median Income Ratio']
                         , name = 'Average'
                         , line=go.scatter.Line(color="#2C3539", dash="dot")
                        ))

title = '<b>Rent to Monthly Median Household Income Ratio</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=0.2, range = [0, 1.1])
fig.update_xaxes(tick0=0, dtick=1, range=[2014.5, 2022.5])
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)